# CatLLM — QLoRA Fine-Tuning + Public Deployment

This notebook:
1. **Loads** your cattle-genetics Q&A Excel (seed + L3 + L4 hierarchy → 7,360 unique pairs)
2. **Trains** a QLoRA adapter on Mistral-7B-Instruct
3. **Evaluates** semantic robustness (cosine similarity vs gold answers)
4. **Serves** the model via vLLM with a **public URL** you can plug into the CatLLM app

**Requirements:** Colab Pro with GPU (L4 or T4). Runtime → Change runtime type → GPU.

## 1. Install Dependencies

In [ ]:
!pip install -q \
    torch transformers trl peft bitsandbytes datasets \
    sentence-transformers accelerate openpyxl \
    vllm pyngrok

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Upload Excel Data

Upload your merged training Excel file when prompted.
Expected file: `catLLM_training_data_L3_merged_with_L4_rowsmatch.xlsx`

In [ ]:
from google.colab import files

uploaded = files.upload()
EXCEL_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {EXCEL_PATH} ({len(uploaded[EXCEL_PATH]) / 1e6:.1f} MB)")

## 3. Load Data — Multi-Turn Chains

Each row in the Excel contains a **full conversation chain**: Seed Q/A → L3 follow-up Q/A → L4 deeper follow-up Q/A.

We build **three types** of training examples:
- **3-turn chains** (seed → L3 → L4): teaches the model to handle deepening conversations
- **2-turn chains** (seed → L3): teaches follow-up reasoning
- **Flat single-turn**: teaches breadth across all levels

In [ ]:
import pandas as pd
import json
import random
import os
from typing import List, Dict, Any, Optional
from collections import Counter


def load_excel_multiturn_chains(excel_path: str, include_flat: bool = True) -> List[Dict[str, Any]]:
    """Build multi-turn conversation chains from the L1→L3→L4 hierarchy.

    Each row in the Excel contains a full chain:
      Seed Q/A  →  L3 Q/A (follow-up)  →  L4 Q/A (deeper follow-up)

    Creates training examples in three formats:
      1) Full 3-turn chains:  system + seed Q/A + L3 Q/A + L4 Q/A  (teaches depth)
      2) 2-turn chains:       system + seed Q/A + L3 Q/A            (teaches follow-up)
      3) Flat single-turn:    system + Q + A  per level             (teaches breadth)
    """
    df = pd.read_excel(excel_path)
    seen = set()
    records = []

    def _make_system(row):
        parts = []
        for key in ["User Role", "Difficulty Level", "Cluster", "Primary Data Source"]:
            if key in df.columns and pd.notna(row.get(key, None)):
                if key == "User Role":
                    parts.append(f"You are acting as: {row[key]}.")
                elif key == "Difficulty Level":
                    parts.append(f"Target difficulty: {row[key]}.")
                elif key == "Cluster":
                    parts.append(f"Topic cluster: {row[key]}.")
                elif key == "Primary Data Source":
                    parts.append(f"Primary data source: {row[key]}.")
        return " ".join(parts).strip()

    def _make_meta(row, chain_type):
        meta = {}
        for k in ["Question ID", "Cluster", "User Role", "Difficulty Level"]:
            if k in df.columns and pd.notna(row.get(k, None)):
                meta[k] = str(row[k])
        meta["chain_type"] = chain_type
        return meta

    def _valid(text):
        s = str(text).strip()
        return bool(s) and s != "nan"

    def _add(messages, chain_type, row):
        dedup_key = tuple((m["role"], m["content"][:200]) for m in messages)
        if dedup_key in seen:
            return
        seen.add(dedup_key)
        records.append({"messages": messages, "metadata": _make_meta(row, chain_type)})

    for _, row in df.iterrows():
        sys_text = _make_system(row)
        seed_q = str(row.get("Question", "")).strip()
        seed_a = str(row.get("Response", "")).strip()
        l3_q = str(row.get("l3_question", "")).strip()
        l3_a = str(row.get("l3_answer", "")).strip()
        l4_q = str(row.get("L4 Question", "")).strip()
        l4_a = str(row.get("L4 Answer", "")).strip()
        base = [{"role": "system", "content": sys_text}] if sys_text else []

        # Full 3-turn chain: seed → L3 → L4
        if _valid(seed_q) and _valid(seed_a) and _valid(l3_q) and _valid(l3_a) and _valid(l4_q) and _valid(l4_a):
            chain3 = base + [
                {"role": "user", "content": seed_q}, {"role": "assistant", "content": seed_a},
                {"role": "user", "content": l3_q}, {"role": "assistant", "content": l3_a},
                {"role": "user", "content": l4_q}, {"role": "assistant", "content": l4_a},
            ]
            _add(chain3, "chain_3turn", row)

        # 2-turn chain: seed → L3
        if _valid(seed_q) and _valid(seed_a) and _valid(l3_q) and _valid(l3_a):
            chain2 = base + [
                {"role": "user", "content": seed_q}, {"role": "assistant", "content": seed_a},
                {"role": "user", "content": l3_q}, {"role": "assistant", "content": l3_a},
            ]
            _add(chain2, "chain_2turn", row)

        # Flat single-turn examples
        if include_flat:
            for level, (q, a) in [("seed", (seed_q, seed_a)), ("l3", (l3_q, l3_a)), ("l4", (l4_q, l4_a))]:
                if _valid(q) and _valid(a):
                    flat = base + [{"role": "user", "content": q}, {"role": "assistant", "content": a}]
                    _add(flat, f"flat_{level}", row)

    return records


# Load with multi-turn chains + flat examples
all_records = load_excel_multiturn_chains(EXCEL_PATH, include_flat=True)
print(f"Total training examples: {len(all_records)}")

# Distribution
types = Counter(r["metadata"]["chain_type"] for r in all_records)
roles = Counter(r["metadata"].get("User Role", "?") for r in all_records)
print(f"\nBy type:")
for t, c in sorted(types.items()):
    print(f"  {t}: {c}")
print(f"\nBy role: {dict(roles)}")

# Show a sample 3-turn chain
sample = next(r for r in all_records if r["metadata"]["chain_type"] == "chain_3turn")
print(f"\n--- Sample 3-turn chain ---")
for m in sample["messages"]:
    content = m["content"][:150] + "..." if len(m["content"]) > 150 else m["content"]
    print(f"[{m['role']:9s}] {content}")

## 4. Train / Eval Split

In [ ]:
SEED = 42
EVAL_RATIO = 0.15  # 15% for eval since we have 7K+ samples
OUTPUT_DIR = "/content/catllm_output"

random.seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

shuffled = all_records[:]
random.shuffle(shuffled)
n_eval = max(1, int(len(shuffled) * EVAL_RATIO))
eval_records = shuffled[:n_eval]
train_records = shuffled[n_eval:]

print(f"Train: {len(train_records)} | Eval: {len(eval_records)}")

# Save for transparency
def save_jsonl(records, path):
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

save_jsonl(train_records, os.path.join(OUTPUT_DIR, "train.jsonl"))
save_jsonl(eval_records, os.path.join(OUTPUT_DIR, "eval.jsonl"))
print(f"Saved to {OUTPUT_DIR}/train.jsonl and eval.jsonl")

## 5. QLoRA Training

Trains a 4-bit quantized Mistral-7B with LoRA adapters. ~30–60 min on L4, ~2–3h on T4.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from trl import SFTTrainer
from peft import LoraConfig
from datasets import Dataset

# --- Config ---
BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
MAX_SEQ_LEN = 1536          # was 2048 — shorter = faster, 3-turn chains fit in ~1400 tokens
EPOCHS = 2                   # was 3 — 2 is enough with 14K examples
LR = 3e-4                   # was 2e-4 — slightly higher to compensate for fewer epochs
BATCH_SIZE = 4               # was 1 — L4 has 24GB, can handle batch=4 with 4-bit
GRAD_ACCUM = 4               # was 16 — effective batch = 4*4=16 (was 1*16=16, same but 4x faster)
LORA_R = 16
LORA_ALPHA = 16

# --- Quantization ---
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=dtype,
)

# --- Load model & tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
print(f"Model loaded: {BASE_MODEL} (4-bit, {dtype})")

# --- Format datasets ---
def to_prompt_text(messages):
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    except Exception:
        return "\n\n".join(f"### {m['role'].upper()}\n{m['content']}" for m in messages)

train_ds = Dataset.from_dict({"text": [to_prompt_text(r["messages"]) for r in train_records]})
eval_ds = Dataset.from_dict({"text": [to_prompt_text(r["messages"]) for r in eval_records]})
print(f"Train samples: {len(train_ds)} | Eval samples: {len(eval_ds)}")

# --- LoRA config ---
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# --- Training args ---
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=400,
    save_total_limit=2,
    bf16=(dtype == torch.bfloat16),
    fp16=(dtype == torch.float16),
    report_to="none",
    gradient_checkpointing=True,
    dataloader_num_workers=2,    # parallel data loading
    torch_compile=False,          # avoid compile overhead for short runs
)

# --- Train ---
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    peft_config=peft_config,
)

print("Starting training...")
trainer.train()

# --- Save adapter ---
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "adapter")
os.makedirs(ADAPTER_DIR, exist_ok=True)
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nAdapter saved to {ADAPTER_DIR}")

## 6. Semantic Evaluation

Generates answers for eval questions and measures cosine similarity against gold answers.

In [ ]:
from sentence_transformers import SentenceTransformer, util
from peft import PeftModel

# Reload model with adapter for eval
eval_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
eval_model = PeftModel.from_pretrained(eval_model, ADAPTER_DIR)
eval_model.eval()

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Evaluate on a subset (full eval set can be slow)
N_EVAL = min(50, len(eval_records))
eval_subset = eval_records[:N_EVAL]

sims = []
examples = []

for i, r in enumerate(eval_subset):
    user = next((m for m in r["messages"] if m["role"] == "user"), None)
    asst = next((m for m in r["messages"] if m["role"] == "assistant"), None)
    if not user or not asst:
        continue

    q, gold = user["content"], asst["content"]
    sys_msg = next((m for m in r["messages"] if m["role"] == "system"), None)
    messages = ([sys_msg] if sys_msg else []) + [{"role": "user", "content": q}]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(eval_model.device)

    with torch.no_grad():
        out = eval_model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=True,
            top_p=0.9,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    embs = embedder.encode([gen, gold], convert_to_tensor=True, normalize_embeddings=True)
    sim = float(util.cos_sim(embs[0], embs[1]).item())
    sims.append(sim)

    if len(examples) < 5:
        examples.append({"question": q[:150], "generated": gen[:300], "gold": gold[:300], "similarity": round(sim, 3)})

    if (i + 1) % 10 == 0:
        print(f"  Evaluated {i+1}/{N_EVAL}  (running avg sim: {sum(sims)/len(sims):.3f})")

avg_sim = sum(sims) / len(sims) if sims else 0
print(f"\n{'='*50}")
print(f"Average semantic similarity: {avg_sim:.3f}  (n={len(sims)})")
print(f"{'='*50}")

# Save results
results = {"avg_similarity": avg_sim, "n": len(sims), "examples": examples}
with open(os.path.join(OUTPUT_DIR, "semantic_eval.json"), "w") as f:
    json.dump(results, f, indent=2)

# Show examples
for ex in examples:
    print(f"\nQ: {ex['question']}")
    print(f"Sim: {ex['similarity']}")
    print(f"Gen: {ex['generated'][:200]}...")

## 7. Quick Inference Test

In [ ]:
test_questions = [
    "What traits have improved the most across Angus cattle in the last 5 years?",
    "How do I read an EPD report for a first-calf heifer?",
    "What is the role of the MSTN gene in beef cattle?",
]

for q in test_questions:
    messages = [
        {"role": "system", "content": "You are CattLLM, a cattle-genetics assistant. Be concise and cite sources."},
        {"role": "user", "content": q},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(eval_model.device)

    with torch.no_grad():
        out = eval_model.generate(
            **inputs, max_new_tokens=400, do_sample=True, top_p=0.9, temperature=0.5,
            pad_token_id=tokenizer.eos_token_id,
        )
    answer = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    print(f"\nQ: {q}")
    print(f"A: {answer[:500]}")
    print("-" * 60)

## 8. Save Adapter to Google Drive

Persist the trained adapter so you don't lose it when the session ends.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/catllm_adapter"
!mkdir -p {DRIVE_DIR}
!cp -r {ADAPTER_DIR}/* {DRIVE_DIR}/
!cp {OUTPUT_DIR}/semantic_eval.json {DRIVE_DIR}/
print(f"Adapter + eval saved to Google Drive: {DRIVE_DIR}")

## 9. Deploy: Serve with vLLM + Public URL

This cell starts a vLLM OpenAI-compatible server with the LoRA adapter and creates a public URL.

**The public URL will be printed below** — paste it into the CatLLM app sidebar under **Local (OpenAI-compatible)**.

In [ ]:
# Free up GPU memory from training/eval before serving
import gc
del model, eval_model, trainer
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared.")

In [ ]:
import subprocess, time, threading, os

VLLM_PORT = 8000

# Start vLLM server in the background
vllm_cmd = [
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", BASE_MODEL,
    "--enable-lora",
    "--lora-modules", f"catllm={ADAPTER_DIR}",
    "--port", str(VLLM_PORT),
    "--dtype", "auto",
    "--max-model-len", "2048",
    "--gpu-memory-utilization", "0.90",
    "--trust-remote-code",
]

print("Starting vLLM server...")
vllm_proc = subprocess.Popen(
    vllm_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Stream logs in background thread
def stream_logs(proc):
    for line in iter(proc.stdout.readline, b""):
        print(f"[vLLM] {line.decode().rstrip()}")

log_thread = threading.Thread(target=stream_logs, args=(vllm_proc,), daemon=True)
log_thread.start()

# Wait for server to be ready
import urllib.request
for i in range(120):  # up to 4 minutes
    try:
        urllib.request.urlopen(f"http://localhost:{VLLM_PORT}/health")
        print(f"\nvLLM server ready on port {VLLM_PORT}!")
        break
    except Exception:
        time.sleep(2)
else:
    print("WARNING: vLLM server did not become ready in time. Check logs above.")

In [ ]:
# --- Create public tunnel for vLLM API ---
# Try ngrok first, fall back to cloudflare

PUBLIC_URL = None

try:
    from pyngrok import ngrok
    NGROK_TOKEN = input("Enter ngrok auth token (get free at dashboard.ngrok.com, or press Enter to skip): ").strip()
    if NGROK_TOKEN:
        ngrok.set_auth_token(NGROK_TOKEN)
        tunnel = ngrok.connect(VLLM_PORT, "http")
        PUBLIC_URL = tunnel.public_url
        print(f"ngrok tunnel active!")
    else:
        raise Exception("skip to cloudflare")
except Exception as e:
    print(f"ngrok unavailable ({e}), trying Cloudflare tunnel...")

if not PUBLIC_URL:
    try:
        !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
        !chmod +x /usr/local/bin/cloudflared
        import subprocess, re
        cf_proc = subprocess.Popen(
            ["cloudflared", "tunnel", "--url", f"http://localhost:{VLLM_PORT}"],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        )
        for _ in range(30):
            line = cf_proc.stdout.readline().decode()
            match = re.search(r"(https://[\w-]+\.trycloudflare\.com)", line)
            if match:
                PUBLIC_URL = match.group(1)
                break
            time.sleep(1)
    except Exception as e:
        print(f"Cloudflare tunnel failed: {e}")

if PUBLIC_URL:
    print(f"\n{'='*60}")
    print(f"vLLM PUBLIC URL: {PUBLIC_URL}/v1")
    print(f"{'='*60}")
else:
    print(f"vLLM running locally at http://localhost:{VLLM_PORT}/v1")


## 10. Test the Public Endpoint

In [ ]:
from openai import OpenAI

# Test against the running server
test_url = PUBLIC_URL + "/v1" if PUBLIC_URL else f"http://localhost:{VLLM_PORT}/v1"
test_client = OpenAI(api_key="local", base_url=test_url)

resp = test_client.chat.completions.create(
    model="catllm",
    messages=[
        {"role": "system", "content": "You are CattLLM, a cattle-genetics assistant."},
        {"role": "user", "content": "What is calving ease and why does it matter for a commercial herd?"},
    ],
    temperature=0.3,
    max_tokens=400,
)

print("Response from deployed model:")
print(resp.choices[0].message.content)

## 12. Deploy Full RAG App from Colab

Now we'll run the **entire CatLLM Streamlit app** on Colab — including document ingestion, FAISS+BM25 indexing, and the chat UI — all powered by your fine-tuned model running on vLLM.

```
This Colab instance:
├── vLLM (port 8000) — your fine-tuned Mistral-7B + LoRA
├── Streamlit (port 8501) — full RAG app
│   ├── 80 PDFs ingested → FAISS + BM25 index
│   ├── Embeddings via OpenAI or sentence-transformers
│   └── Chat answers from vLLM (localhost:8000)
└── Public URL → you open it in any browser
```

In [ ]:
# --- Clone the CatLLM repo (includes app code + CLUSTERS/ PDFs) ---
!git clone https://github.com/iamkani/CatLLM.git /content/CatLLM 2>/dev/null || echo "Already cloned"
!pip install -q -r /content/CatLLM/requirements.txt

import sys
sys.path.insert(0, "/content/CatLLM")

# Verify
from catllm.llm import ensure_client
print("CatLLM app code loaded successfully")
!ls /content/CatLLM/CLUSTERS/ | head -5
print("...")

### 12a. Configure Embeddings

Choose how to embed documents. OpenAI gives best quality; sentence-transformers is free but lower quality.

In [ ]:
import os
import getpass

# --- Choose embedding method ---
# Option 1: OpenAI (better quality, costs a few cents)
# Option 2: Local sentence-transformers (free, no API key needed)

USE_OPENAI_EMBEDDINGS = True  # Set to False for free local embeddings

if USE_OPENAI_EMBEDDINGS:
    api_key = getpass.getpass("Enter your OpenAI API key (or press Enter to use local embeddings): ")
    if api_key.strip():
        os.environ["OPENAI_API_KEY"] = api_key.strip()
        EMBED_PROVIDER = "OpenAI"
        EMBED_MODEL = "text-embedding-3-large"
        print(f"Using OpenAI embeddings: {EMBED_MODEL}")
    else:
        USE_OPENAI_EMBEDDINGS = False

if not USE_OPENAI_EMBEDDINGS:
    # Use local sentence-transformers for embeddings
    # We'll create a wrapper that makes it compatible with the app's embed_texts interface
    EMBED_PROVIDER = "Local (OpenAI-compatible)"
    EMBED_MODEL = "catllm"  # vLLM will handle this if the model supports embeddings
    os.environ["LOCAL_API_BASE"] = f"http://localhost:{VLLM_PORT}/v1"
    os.environ["LOCAL_API_KEY"] = "local"
    print(f"Using local embeddings via vLLM at localhost:{VLLM_PORT}")

# Always use fine-tuned model for chat
CHAT_PROVIDER = "Local (OpenAI-compatible)"
CHAT_MODEL = "catllm"
os.environ["LOCAL_API_BASE"] = f"http://localhost:{VLLM_PORT}/v1"
os.environ["LOCAL_API_KEY"] = "local"

print(f"\nChat: {CHAT_MODEL} via vLLM (localhost:{VLLM_PORT})")
print(f"Embeddings: {EMBED_MODEL} via {EMBED_PROVIDER}")

### 12b. Pre-Build RAG Index (headless)

Ingest all ~80 PDFs from CLUSTERS/, chunk them, embed, and build FAISS+BM25 indexes.
This runs the full pipeline programmatically so the Streamlit app launches with a ready index.

In [ ]:
import sys, numpy as np
sys.path.insert(0, "/content/CatLLM")

from catllm.ingest import ingest_folder
from catllm.utils_text import chunk_text_smart
from catllm.tagging import tag_text_for_meta
from catllm.embeddings import embed_texts
from catllm.indexer import build_faiss_index, build_bm25_index
from catllm.persistence import save_store
from catllm.llm import ensure_client

CLUSTERS_PATH = "/content/CatLLM/CLUSTERS"
STORE_DIR = "/content/CatLLM/.rag_store"
CHUNK_SIZE = 900
OVERLAP = 120

# --- Step 1: Ingest PDFs ---
print("Ingesting PDFs from CLUSTERS/...")
docs, fails = ingest_folder(
    CLUSTERS_PATH,
    exts=["pdf", "txt", "md", "csv"],
    recursive=True,
    max_files=200,
    max_size_mb=50,
    seen_hashes=set(),
)
print(f"  Ingested {len(docs)} documents ({len(fails)} failed)")

# --- Step 2: Chunk + tag ---
print("Chunking and tagging...")
chunks = []
for item in docs:
    title, text, extra = item if len(item) == 3 else (item[0], item[1], {})
    for ch in chunk_text_smart(text, CHUNK_SIZE, OVERLAP):
        meta = {"title": title, "source": title}
        meta.update(extra)
        meta.update(tag_text_for_meta(ch))
        chunks.append({"text": ch, "meta": meta})
print(f"  {len(chunks)} chunks created")

# --- Step 3: Embed ---
print(f"Embedding {len(chunks)} chunks with {EMBED_PROVIDER} / {EMBED_MODEL}...")
client = ensure_client(EMBED_PROVIDER, base_url=os.environ.get("LOCAL_API_BASE", ""))
texts = [c["text"] for c in chunks if c["text"].strip()]
embs = embed_texts(client, texts, EMBED_MODEL)
print(f"  Embeddings shape: {embs.shape}")

# --- Step 4: Build indexes ---
print("Building FAISS + BM25 indexes...")
faiss_index = build_faiss_index(embs)
bm25_index = build_bm25_index(texts)

# --- Step 5: Save ---
save_store(STORE_DIR, chunks, embs, faiss_index)
print(f"  Saved to {STORE_DIR}")

# Also save BM25 corpus texts for later rebuild
import json
with open(os.path.join(STORE_DIR, "bm25_texts.json"), "w") as f:
    json.dump(texts, f)

print(f"\nRAG index ready: {len(chunks)} chunks, {embs.shape[1]}-dim embeddings")
print(f"Clusters found: {len(set(c['meta'].get('cluster','') for c in chunks if c['meta'].get('cluster')))}")

### 12c. Launch Streamlit App + Public URL

This starts the full CatLLM app and creates a public URL you can open in your browser.

In [ ]:
import subprocess, time, re

STREAMLIT_PORT = 8501

# --- Set environment for the Streamlit app ---
env = os.environ.copy()
env["PROVIDER"] = "Local (OpenAI-compatible)"
env["LOCAL_API_BASE"] = f"http://localhost:{VLLM_PORT}/v1"
env["LOCAL_API_KEY"] = "local"
env["STORE_DIR"] = STORE_DIR
env["AUTOSAVE"] = "false"  # index is pre-built

# --- Launch Streamlit in background ---
print("Starting Streamlit app...")
st_proc = subprocess.Popen(
    [
        "streamlit", "run", "/content/CatLLM/app.py",
        "--server.port", str(STREAMLIT_PORT),
        "--server.headless", "true",
        "--server.address", "0.0.0.0",
        "--browser.gatherUsageStats", "false",
    ],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    cwd="/content/CatLLM",
)

# Wait for Streamlit to be ready
for _ in range(30):
    try:
        urllib.request.urlopen(f"http://localhost:{STREAMLIT_PORT}")
        print(f"Streamlit app ready on port {STREAMLIT_PORT}")
        break
    except Exception:
        time.sleep(2)
else:
    print("WARNING: Streamlit did not start. Check logs.")

# --- Create public tunnel for Streamlit ---
APP_URL = None

# Try ngrok (reuse token if already set)
try:
    from pyngrok import ngrok
    tunnel = ngrok.connect(STREAMLIT_PORT, "http")
    APP_URL = tunnel.public_url
except Exception:
    pass

# Fallback: cloudflare
if not APP_URL:
    try:
        cf_st = subprocess.Popen(
            ["cloudflared", "tunnel", "--url", f"http://localhost:{STREAMLIT_PORT}"],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        )
        for _ in range(30):
            line = cf_st.stdout.readline().decode()
            match = re.search(r"(https://[\w-]+\.trycloudflare\.com)", line)
            if match:
                APP_URL = match.group(1)
                break
            time.sleep(1)
    except Exception:
        pass

print(f"\n{'='*60}")
if APP_URL:
    print(f"CATLLM APP URL: {APP_URL}")
else:
    print(f"App running at: http://localhost:{STREAMLIT_PORT}")
print(f"{'='*60}")
print(f"\nOpen this URL in your browser.")
print(f"The app is pre-loaded with:")
print(f"  - {len(chunks)} document chunks from CLUSTERS/")
print(f"  - FAISS + BM25 indexes ready")
print(f"  - Fine-tuned CatLLM model for chat")
print(f"\nIn the sidebar:")
print(f"  - Provider is set to Local (OpenAI-compatible)")
print(f"  - Click 'Load saved' to load the pre-built index")
print(f"  - Then start chatting!")

## 13. Keep Everything Running

Run this cell to keep both vLLM and Streamlit alive. The public URL stays active as long as this cell is running.

**To stop:** Press the stop button.

In [ ]:
import time, urllib.request

print(f"CatLLM App:  {APP_URL or f'http://localhost:{STREAMLIT_PORT}'}")
print(f"vLLM API:    {PUBLIC_URL}/v1" if PUBLIC_URL else f"vLLM API: http://localhost:{VLLM_PORT}/v1")
print(f"\nBoth servers running. Press stop button to shut down.\n")

try:
    while True:
        time.sleep(60)
        # Health checks
        vllm_ok = st_ok = False
        try:
            urllib.request.urlopen(f"http://localhost:{VLLM_PORT}/health")
            vllm_ok = True
        except Exception:
            pass
        try:
            urllib.request.urlopen(f"http://localhost:{STREAMLIT_PORT}")
            st_ok = True
        except Exception:
            pass
        status = f"[{time.strftime('%H:%M')}] vLLM: {'OK' if vllm_ok else 'DOWN'}  |  Streamlit: {'OK' if st_ok else 'DOWN'}"
        print(status, end="\r")
        if not vllm_ok:
            print(f"\n  WARNING: vLLM server may have stopped!")
        if not st_ok:
            print(f"\n  WARNING: Streamlit app may have stopped!")
except KeyboardInterrupt:
    print("\nShutting down...")
    vllm_proc.terminate()
    st_proc.terminate()
    print("All servers stopped.")